# Unemployment Analysis in India
## OASIS INFOBYTE SIP - Data Science Task 2

This notebook analyses monthly unemployment data across Indian states and regions, including changes around the COVID-19 lockdown.

**Source:** Kaggle, *Unemployment in India* (Gokul Raj Kuppan). The data originates from CMIE, as described on the dataset page.

## 1. Imports and data loading

The supplied CSV is kept beside this notebook so the analysis can be rerun without changing the file path.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

DATA_FILE = 'Unemployment in India.csv'
raw_df = pd.read_csv(DATA_FILE)
raw_df.head()

## 2. Data inspection and cleaning

The raw file has leading spaces in several column names and blank separator rows. The next cell standardises names, removes empty rows, converts the date field, and confirms the resulting data types.

In [ ]:
print(f'Raw shape: {raw_df.shape}')
print('\nNull values in raw data:')
display(raw_df.isnull().sum().to_frame('null_count'))

df = raw_df.copy()
df.columns = df.columns.str.strip()
df = df.dropna(how='all').copy()
text_columns = df.select_dtypes(include=['object', 'string']).columns
df[text_columns] = df[text_columns].apply(lambda column: column.str.strip())
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')
numeric_columns = [
    'Estimated Unemployment Rate (%)',
    'Estimated Employed',
    'Estimated Labour Participation Rate (%)'
]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=['Region', 'Date'] + numeric_columns).copy()
df = df.sort_values('Date').reset_index(drop=True)

print(f'Clean shape: {df.shape}')
print(f'Date range: {df["Date"].min():%d %b %Y} to {df["Date"].max():%d %b %Y}')
display(df.dtypes.to_frame('data_type'))
df.head()

## 3. Region-wise average unemployment rate

This comparison highlights which states had the highest average unemployment rate across the period covered by the dataset. Averages should be interpreted across the available monthly observations, rather than as a population-weighted national rate.

In [ ]:
state_avg = (df.groupby('Region', as_index=False)['Estimated Unemployment Rate (%)']
              .mean()
              .sort_values('Estimated Unemployment Rate (%)', ascending=False))

plt.figure(figsize=(11, 8))
sns.barplot(data=state_avg.head(10), y='Region', x='Estimated Unemployment Rate (%)', hue='Region', legend=False, palette='Reds_r')
plt.title('Top 10 States/Regions by Average Unemployment Rate')
plt.xlabel('Average unemployment rate (%)')
plt.ylabel('State / region')
plt.tight_layout()
plt.show()

print(f"Observation: {state_avg.iloc[0]['Region']} has the highest average unemployment rate ({state_avg.iloc[0]['Estimated Unemployment Rate (%)']:.2f}%).")

## 4. Month-wise unemployment trend

A monthly average makes the broad movement in unemployment easier to see. Because this is an average of state-level observations, spikes show widespread worsening or particularly large state-level changes in that month.

In [ ]:
monthly_avg = (df.groupby('Date', as_index=False)['Estimated Unemployment Rate (%)']
                .mean()
                .sort_values('Date'))

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_avg, x='Date', y='Estimated Unemployment Rate (%)', marker='o', linewidth=2)
plt.axvline(pd.Timestamp('2020-03-25'), color='crimson', linestyle='--', label='India COVID-19 lockdown')
plt.title('Average Unemployment Rate Over Time')
plt.xlabel('Month')
plt.ylabel('Average unemployment rate (%)')
plt.legend()
plt.tight_layout()
plt.show()

peak = monthly_avg.loc[monthly_avg['Estimated Unemployment Rate (%)'].idxmax()]
print(f"Observation: the monthly average peaks at {peak['Estimated Unemployment Rate (%)']:.2f}% in {peak['Date']:%B %Y}.")

## 5. Time-series comparison for major states

Maharashtra, Tamil Nadu, and Karnataka are included as large state economies. Comparing their paths shows whether the timing and size of unemployment changes were similar across major regions.

In [ ]:
major_states = ['Maharashtra', 'Tamil Nadu', 'Karnataka']
available_states = [state for state in major_states if state in df['Region'].unique()]
if len(available_states) < 3:
    available_states = state_avg['Region'].head(3).tolist()

state_monthly = (df[df['Region'].isin(available_states)]
                 .groupby(['Date', 'Region'], as_index=False)['Estimated Unemployment Rate (%)']
                 .mean())

plt.figure(figsize=(12, 6))
sns.lineplot(data=state_monthly, x='Date', y='Estimated Unemployment Rate (%)', hue='Region', marker='o', linewidth=2)
plt.axvline(pd.Timestamp('2020-03-25'), color='black', linestyle='--', alpha=0.7, label='COVID-19 lockdown')
plt.title('Unemployment Rate Over Time: Three Major States')
plt.xlabel('Month')
plt.ylabel('Unemployment rate (%)')
plt.legend(title='State / region')
plt.tight_layout()
plt.show()

print('Observation: the lines show that COVID-period disruption was not uniform; each state has a distinct level and month-to-month pattern.')

## 6. Correlation between labour-market measures

The supplied file contains **Estimated Employed** (a count), not an employment-rate percentage. The heatmap therefore uses that available employment measure alongside unemployment and labour participation; correlation measures association, not causation.

In [ ]:
correlation_data = df[numeric_columns].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_data, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Correlation Between Unemployment, Employment, and Labour Participation')
plt.tight_layout()
plt.show()

print('Observation: positive values indicate variables that tend to move together, while negative values indicate opposite movement in this dataset.')

## 7. Pre-COVID vs. post-COVID comparison

The comparison uses 25 March 2020, the start of India's nationwide COVID-19 lockdown, as the cutoff. Monthly observations before this date are pre-COVID and those on or after it are post-COVID.

In [ ]:
covid_cutoff = pd.Timestamp('2020-03-25')
df['Period'] = df['Date'].ge(covid_cutoff).map({True: 'Post-COVID', False: 'Pre-COVID'})
period_comparison = (df.groupby('Period', as_index=False)['Estimated Unemployment Rate (%)']
                     .mean()
                     .set_index('Period')
                     .reindex(['Pre-COVID', 'Post-COVID'])
                     .reset_index())

display(period_comparison.round({'Estimated Unemployment Rate (%)': 2}))
plt.figure(figsize=(7, 5))
sns.barplot(data=period_comparison, x='Period', y='Estimated Unemployment Rate (%)', hue='Period', legend=False, palette=['#4C78A8', '#E45756'])
plt.title('Mean Unemployment Rate: Pre-COVID vs. Post-COVID')
plt.xlabel('Period')
plt.ylabel('Mean unemployment rate (%)')
plt.tight_layout()
plt.show()

pre_rate = period_comparison.loc[period_comparison['Period'].eq('Pre-COVID'), 'Estimated Unemployment Rate (%)'].iloc[0]
post_rate = period_comparison.loc[period_comparison['Period'].eq('Post-COVID'), 'Estimated Unemployment Rate (%)'].iloc[0]
difference = post_rate - pre_rate
print(f'Observation: average unemployment was {pre_rate:.2f}% before COVID-19 and {post_rate:.2f}% after the cutoff, a change of {difference:+.2f} percentage points.')

## 8. Conclusion

The analysis meets the task requirements through data cleaning, regional and monthly EDA, three-state time-series comparison, a top-10 bar chart, a correlation heatmap, and a pre/post-COVID comparison. The charts should be interpreted within the dataset's monthly coverage and state-level aggregation.